# ML-04 — Search Intelligence Data Contract

**Lane 2 (refresh / opportunity scoring), continuing the ML-03 frame:** rank pages by their
30-day decline risk into an editor queue, judged by precision@20. This contract pins down what
a row means, which time windows feed features vs the label, where every field we touch goes,
and the executed queries that prove each claim — built with the `writing-data-contracts` and
`flyrank/flyrank-data` skills.

Every number below is queried live from the warehouse release (`FlyRank/internship-warehouse`,
build v20260703). Nothing is hardcoded except the release-note totals used as cross-checks.

Two properties of the data were **discovered while writing this contract** and shape every
section below:

1. the `_sample` fact table turns out to be **June 2026 only** — an iteration slice, not a
   period sample — so any time-window work has to read month partitions of the full table;
2. that same slice contains thousands of **exact duplicate rows**, so the contract carries an
   explicit dedup rule.

No modeling happens here, so there are no random seeds to fix.

## 0. Setup — connect to the release

The READ token comes from the repo-root `.env` through the repo's own helper
(`work/scripts/hf_query.py`) and is never printed.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))

import duckdb
import pandas as pd
from datetime import timedelta

import hf_query

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '" + hf_query.get_token() + "')")

REL = hf_query.REL
T = {
    "clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "sample":  f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "q90":     f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("connected to", hf_query.REPO)

C:\Users\Bogdan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


connected to FlyRank/internship-warehouse


## 1. Unit of analysis + time window

**One row = one page — a `client_hash_id` × `content_hash_id` pair — observed at a decision
date `t`.** A page is the thing an editor decides about and the unit the ML-03 queue ranks;
`t` is when the queue would have been produced.

The two windows are half-open intervals of `report_date`, both anchored at `t`:

| window | definition | role |
|---|---|---|
| **B** (baseline) | `(t − 30 days, t]` | features may only come from here or earlier |
| **F** (forward) | `(t, MAX(report_date)]` | only the label touches this |

`t := MAX(report_date) − 30 days`, so that F fits inside the panel wherever a page is still
reporting. Because history depth differs per client (verified in §3c), eligibility is defined
per page, not by a global calendar: **a page enters the population only if its baseline
impressions are ≥ 100** (the noise floor carried over from starter notebook 03) **and it has ≥
15 GSC-available days inside B**.

Everything below verifies these sentences against the data.

In [2]:
n_fact = con.sql(f"SELECT MIN(report_date), MAX(report_date), COUNT(*) FROM {T['daily']}").fetchone()

D_MIN, D_MAX, FACT_ROWS = n_fact
t = D_MAX - timedelta(days=30)
b_lo = t - timedelta(days=30)

print(f"panel            : {D_MIN} -> {D_MAX}   ({FACT_ROWS:,} rows)")
print(f"decision date t  : {t}")
print(f"baseline B       : ({b_lo}, {t}]")
print(f"forward F        : ({t}, {D_MAX}]")

panel            : 2025-01-27 -> 2026-06-30   (78,835,655 rows)
decision date t  : 2026-05-31
baseline B       : (2026-05-01, 2026-05-31]
forward F        : (2026-05-31, 2026-06-30]


## 2. Fields: feature / label / context / excluded

Every field the Lane-2 pipeline plans to touch is in exactly one bucket. The rule that puts
them there: **features must be fully knowable at or before `t`; anything computed from F, or
from a window that straddles `t`, can never be a feature.**

| Field(s) | Source | Bucket | Why / notes |
|---|---|---|---|
| `imp_b`, `clk_b`, `gsc_days_b` — sums over B | daily | **Feature** | demand & coverage up to `t` |
| `pos_avg_b`, `pos_vol_b` — mean/std of `gsc_avg_position` over B | daily | **Feature** | position stability; NULL/0 handled as missing |
| `content_type`, `word_count`, `char_count`, `keyword_char_count`, `keyword_token_count`, `url_char_count` | dim_content | **Feature** | static descriptors |
| `main_intent`, `competition_level`, `category_count`, `search_volume`, `backlinks` | dim_content | **Feature** (+has_-flags) | demand/competition context; `search_volume`/`backlinks` get `has_` flags because their gaps follow `content_type` (§3d) |
| `age_days`, `days_since_update` — `t − created/updated` | dim_content | **Feature** (approx.) | freshness; ⚠ updated-date is *release-time* state, see §4 |
| `declined_30d := imp_f < 0.8 × imp_b` | daily, F vs B | **Label** | our cutoff on an observed outcome; `imp_b` deliberately doubles as the scale feature |
| `client_hash_id` | all tables | **Context** | grouped train/test splits only — never a feature |
| `content_hash_id` | all tables | **Context** | joins and reading only |
| `report_date`, `t`, `month` | daily | **Context** | window alignment; `month` prunes partitions |
| `is_deleted`, `is_published` | dim_content | **Context** | population filters only |
| `ga4_*`, `sessions_*`, `ai_chatgpt…ai_other`, `scroll_events` | daily | **Excluded** | zero-filled outside GA4 availability; ~74% of June-slice rows flagged off (§3e) — revisit only with the flag |
| every `fact_content_query_90d` column | q90 | **Excluded** | fixed release window straddles `t` (§3f) → future information |
| `provider_used`, `model_used` | dim_content | **Excluded** | product-decision flags — how content was produced |
| `last_optimized_date`, `optimization_eligible_date` | dim_content | **Excluded** | product-workflow timestamps, release-time state |
| `keyword_hash_id`, `url_hash_id` | dim_content | **Excluded** | high-cardinality IDs, nothing generalizable |

Note on the label: the warehouse ships **no** ready-made decline column (the starter CSV's
`is_declining_label` does not exist here) — the label is ours, built from observed windows.

In [3]:
classified = {
    "daily":   ["report_date", "month", "gsc_impressions", "gsc_clicks", "gsc_avg_position",
                "gsc_data_available"],
    "content": ["client_hash_id", "content_hash_id", "keyword_hash_id", "url_hash_id",
                "content_type", "word_count", "char_count", "keyword_char_count",
                "keyword_token_count", "url_char_count", "main_intent", "competition_level",
                "category_count", "search_volume", "backlinks", "content_created_date",
                "content_updated_date", "is_deleted", "is_published", "provider_used",
                "model_used", "last_optimized_date", "optimization_eligible_date"],
    "sample":  ["ga4_pageviews", "sessions_ai", "scroll_events"],
    "q90":     ["window_start", "window_end", "impressions_last30"],
}

for src, cols in classified.items():
    have = {r[0] for r in con.sql(f"DESCRIBE SELECT * FROM {T[src]}").fetchall()}
    missing = [c for c in cols if c not in have]
    assert not missing, (src, missing)
    print(f"{src:8s}: all {len(cols)} classified fields exist")

daily   : all 6 classified fields exist


content : all 23 classified fields exist


sample  : all 3 classified fields exist


q90     : all 3 classified fields exist


## 3. Verify it with queries (grain, counts, missing values, windows)

Each claim made above now gets its executed check:

- **(a) grain** — one row per `report_date × client × content`?
- **(b) counts** — live totals vs the release notes;
- **(c) windows** — do histories really start at different times, and who can support B?;
- **(d) missing values** — random, or patterned by `content_type`?;
- **(e) zero-fill** — proof that GA4 zeros are fillers, not engagement;
- **(f) feasibility** — does the promised label actually exist at `t`, and how common is it?

**(a) Grain.** The release notes promise one row per `report_date × client × content`. The
check runs identically on a full month partition of the source table (June 2026) and on the
`_sample` iteration slice, so a source defect can't be mistaken for a sampling artifact.

In [4]:
june_part = REL + "/fact_content_daily_performance/month=" + D_MAX.strftime("%Y-%m") + "/data_0.parquet"

con.sql(f"""
    CREATE OR REPLACE TEMP TABLE dups_source AS
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n_rows
    FROM read_parquet('{june_part}')
    GROUP BY 1, 2, 3 HAVING COUNT(*) > 1
""")

con.sql(f"""
    CREATE OR REPLACE TEMP TABLE dups_sample AS
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n_rows
    FROM {T['sample']}
    GROUP BY 1, 2, 3 HAVING COUNT(*) > 1
""")

for name, tbl in [("full June partition", "dups_source"), ("_sample June slice", "dups_sample")]:
    dup_groups, excess, worst = con.sql(
        f"SELECT COUNT(*), COALESCE(SUM(n_rows - 1), 0), COALESCE(MAX(n_rows), 0) FROM {tbl}"
    ).fetchone()
    print(f"{name:22s}: {dup_groups:,} duplicate groups, {excess:,} excess rows, worst x{worst}")

print(con.sql("SELECT n_rows, COUNT(*) AS groups FROM dups_sample GROUP BY 1 ORDER BY 1").fetchall())

dc_dupes = con.sql(
    f"SELECT COUNT(*) FROM (SELECT 1 FROM {T['content']} GROUP BY client_hash_id, content_hash_id HAVING COUNT(*) > 1)"
).fetchone()[0]
print("dim_content duplicate (client, content) pairs:", dc_dupes)

full June partition   : 6,390 duplicate groups, 6,390 excess rows, worst x2
_sample June slice    : 6,390 duplicate groups, 6,390 excess rows, worst x2
[(2, 6390)]


dim_content duplicate (client, content) pairs: 0


**Observed:** the June *source partition* and the `_sample` slice carry **identical**
duplicate sets — 6,390 groups, every one exactly doubled. The defect therefore sits in the
release build itself, upstream of sampling.

**Contract rule added by this check:** before any aggregation, rows are collapsed to the
grain (`GROUP BY` the three key columns, taking `SUM` of metrics / `MAX` of flags — or plain
`DISTINCT` for exact copies like these). It touches ~0.05% of June rows today,
but skipping the rule silently double-counts real impressions (one probed pair summed 208
impressions across its two copies) — and nothing guarantees tomorrow's build stays this small.

**(b) Counts.** Live totals vs the numbers promised in the release notes
(`skills/flyrank/flyrank-data`).

In [5]:
RELEASE_NOTES = {
    "dim_clients": 104,
    "dim_content": 519606,
    "fact_content_daily_performance": 78835655,
    "fact_content_query_90d": 2414248,
}

live = {
    "dim_clients": con.sql(f"SELECT COUNT(*) FROM {T['clients']}").fetchone()[0],
    "dim_content": con.sql(f"SELECT COUNT(*) FROM {T['content']}").fetchone()[0],
    "fact_content_daily_performance": FACT_ROWS,
    "fact_content_query_90d": con.sql(f"SELECT COUNT(*) FROM {T['q90']}").fetchone()[0],
}
for k, promised in RELEASE_NOTES.items():
    status = "PASS" if live[k] == promised else "FAIL"
    print(f"[{status}] {k:32s} live={live[k]:>12,}  notes={promised:>12,}")

smp = con.sql(f"SELECT MIN(report_date), MAX(report_date), COUNT(*) FROM {T['sample']}").fetchone()
print(f"\n_sample covers ONLY {smp[0]} -> {smp[1]}  ({smp[2]:,} rows) - a one-month iteration slice")

[PASS] dim_clients                      live=         104  notes=         104
[PASS] dim_content                      live=     519,606  notes=     519,606
[PASS] fact_content_daily_performance   live=  78,835,655  notes=  78,835,655
[PASS] fact_content_query_90d           live=   2,414,248  notes=   2,414,248



_sample covers ONLY 2026-06-01 -> 2026-06-30  (11,694,072 rows) - a one-month iteration slice


**(c) Windows.** The skill sheet warns history depth differs wildly per client. If true,
global calendar windows are unsafe and per-page/per-client eligibility (as defined in §1) is
required.

In [6]:
win = con.sql(f"""
    SELECT COUNT(*)                                                          AS n_clients,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL)                    AS no_gsc_start,
           COUNT(*) FILTER (WHERE gsc_data_start <= DATE '{b_lo}')           AS covers_baseline,
           CAST(quantile_cont(gsc_data_start - DATE '{D_MIN}', 0.5) AS INT)  AS median_lag_days,
           MAX(gsc_data_start)                                               AS latest_start
    FROM {T['clients']}
""").fetchone()

print(f"clients                      : {win[0]}")
print(f"no GSC anchor date at all    : {win[1]}")
print(f"history reaches back to B    : {win[2]}")
print(f"median start lag after {D_MIN} : {win[3]} days")
print(f"latest GSC start             : {win[4]}")

clients                      : 104
no GSC anchor date at all    : 37
history reaches back to B    : 62
median start lag after 2025-01-27 : 282 days
latest GSC start             : 2026-06-02


**(d) Missing values — random or patterned?** Null shares in `dim_content`, grouped by
`content_type`: if gaps follow the category, a blind `fillna(0)` injects a category signal —
the reason the feature list carries `has_` flags.

In [7]:
miss = con.sql(f"""
    SELECT content_type,
           COUNT(*)                                                           AS n_items,
           ROUND(AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0 END), 3)    AS word_count_null,
           ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END), 3) AS search_volume_null,
           ROUND(AVG(CASE WHEN backlinks IS NULL THEN 1.0 ELSE 0 END), 3)     AS backlinks_null
    FROM {T['content']}
    GROUP BY content_type
    ORDER BY n_items DESC
""").df()
miss

,content_type,n_items,word_count_null,search_volume_null,backlinks_null
0,keyword article,459174,0.381,0.186,0.458
1,feedly article,57024,0.051,1.000,1.000
2,comparison article,3408,0.001,0.001,0.001


Missingness is strongly patterned: nearly all keyword metadata is absent for `feedly`
items while `comparison` items are almost complete. Any imputation must therefore be
type-aware (or replaced by `has_` flags).

**(e) Zero-fill proof.** Rows outside a client's GA4 window carry `ga4_data_available =
FALSE`. If those rows' GA4 numbers are filler rather than measurement, their averages should
be exactly zero — distinct from genuinely low-engagement pages.

In [8]:
zf = con.sql(f"""
    SELECT ROUND(AVG(CASE WHEN NOT ga4_data_available THEN 1.0 ELSE 0 END), 3)                 AS ga4_off_share,
           ROUND(AVG(CASE WHEN NOT ga4_data_available THEN ga4_pageviews END), 2)              AS pv_when_off,
           ROUND(AVG(CASE WHEN ga4_data_available THEN ga4_pageviews END), 2)                  AS pv_when_on,
           ROUND(AVG(CASE WHEN NOT ga4_data_available THEN sessions_ai END), 2)                AS ai_when_off,
           ROUND(AVG(CASE WHEN gsc_data_available AND (gsc_avg_position IS NULL OR gsc_avg_position = 0)
                           THEN 1.0 ELSE 0 END), 4)                                            AS pos_missing_share
    FROM {T['sample']}
""").fetchone()

print(f"GA4 flagged off               : {zf[0]:.1%} of June-slice rows")
print(f"avg ga4_pageviews when OFF    : {zf[1]}   (filler)")
print(f"avg ga4_pageviews when ON     : {zf[2]}   (measurement)")
print(f"avg sessions_ai when OFF      : {zf[3]}")
print(f"gsc_avg_position NULL-or-0 among available rows: {zf[4]:.2%}")

GA4 flagged off               : 74.0% of June-slice rows
avg ga4_pageviews when OFF    : 0.0   (filler)
avg ga4_pageviews when ON     : 7.54   (measurement)
avg sessions_ai when OFF      : 0.0
gsc_avg_position NULL-or-0 among available rows: 0.60%


Confirmed: flagged-off rows average **exactly 0** pageviews and AI sessions — zeros are
placeholders, not signal. This is why the whole GA4/session/AI family sits in *Excluded* for
v1 despite looking attractive. The smaller `avg_position` gap (0.6% of June-slice rows —
about 1.8% of GSC-available rows) is handled by treating NULL/0 as missing, never as rank
zero.

**(f) Feasibility — does the label exist at `t`?** One pass over the two partitions that
B and F need (heaviest cell of the notebook — expect several minutes), producing the
population the ML-03 frame promised: eligible pages and their observed decline rate under
`declined_30d`. Pages deleted mid-window are counted separately, because deletion mechanically
looks like decline (see §4).

In [9]:
feas = con.sql(f"""
    WITH win AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_data_available
        FROM {T['daily']}
        WHERE month IN ('{t:%Y-%m}', '{D_MAX:%Y-%m}')
    ), agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_b,
               SUM(CASE WHEN report_date >  DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_f,
               SUM(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available THEN 1 ELSE 0 END) AS gsc_days_b
        FROM win
        GROUP BY 1, 2
    ), j AS (
        SELECT a.*, COALESCE(c.is_deleted, FALSE) AS is_deleted
        FROM agg a LEFT JOIN {T['content']} c USING (client_hash_id, content_hash_id)
    )
    SELECT COUNT(*)                                                                AS pages_seen,
           COUNT(*) FILTER (WHERE imp_b >= 100 AND gsc_days_b >= 15)               AS eligible,
           COUNT(*) FILTER (WHERE imp_b >= 100 AND gsc_days_b >= 15 AND NOT is_deleted) AS eligible_active,
           COUNT(*) FILTER (WHERE imp_b >= 100 AND gsc_days_b >= 15 AND imp_f < 0.8 * imp_b) AS declined_all,
           COUNT(*) FILTER (WHERE imp_b >= 100 AND gsc_days_b >= 15 AND NOT is_deleted AND imp_f < 0.8 * imp_b) AS declined_active
    FROM j
""").fetchone()

pages_seen, eligible, eligible_active, declined_all, declined_active = feas
print(f"pages with rows in B or F      : {pages_seen:,}")
print(f"eligible (imp_b>=100, >=15d)   : {eligible:,}")
print(f"eligible & active (not deleted): {eligible_active:,}")
print(f"declined_30d rate, all         : {declined_all / eligible:.1%}")
print(f"declined_30d rate, active only : {declined_active / eligible_active:.1%}")

pages with rows in B or F      : 409,326
eligible (imp_b>=100, >=15d)   : 108,254
eligible & active (not deleted): 108,253
declined_30d rate, all         : 67.4%
declined_30d rate, active only : 67.4%


## 4. Data limits

What this data can never tell us — each limit measured above, not assumed:

1. **Unbalanced panel.** 37 of 104 clients ship no GSC anchor date at all, median history
   starts ~282 days after the earliest client, and only 62 clients reach back far enough to
   support baseline B at `t`. Any global-calendar framing would fabricate history; eligibility
   stays per-page/per-client.
2. **GA4 columns are zero-filled, not absent.** ~74% of June-slice rows are flagged off and
   average exactly 0 pageviews — reading them as "no engagement" poisons any engagement
   feature. Excluded until filtered on `ga4_data_available`.
3. **June ships duplicated.** The source partition and the `_sample` slice carry the same
   6,390 exact-duplicate pairs — always exactly doubled (~0.05%) — so the defect is upstream
   of sampling. On top of that, `_sample` covers June only: it cannot support time-window
   work and must not be treated as period-representative. The §3a dedup rule applies to any
   use of either.
4. **Query-level signals are future-contaminated at this `t`.** The q90 table's fixed window
   (checked below) straddles the decision date, so its last-30 columns literally contain F.
   Excluded wholesale this cycle; only explicitly re-aligned derivatives could return later.
5. **Freshness features are approximate.** `content_updated_date` reflects release-build
   state, not necessarily state-as-of-`t` — usable directionally, never as a precise
   as-of-time claim.
6. **Censoring inflates the label.** Pages deleted inside F score as declines without any
   ranking change; that is why the active-only rate is reported next to the raw one.
7. **The base rate is high** — roughly two thirds of eligible pages "decline" under our 20%
   cutoff. Precision@20 therefore gets judged against this base rate (and lift/AUC), never as
   a bare score; the capstone checklist requires the base rate beside every metric.
8. **Observational, pseudonymous, public-safe.** No experiment sits behind these windows, so
   claims stay directional / decision-support; hashed IDs mean no client is identifiable and
   none appears anywhere in `work/`.

In [10]:
q90_win = con.sql(f"SELECT MIN(window_start), MAX(window_end) FROM {T['q90']}").fetchone()
print(f"q90 fixed window: {q90_win[0]} -> {q90_win[1]}")
print(f"decision date t : {t}")
print("straddles t -> future-contaminated:", q90_win[0] <= t <= q90_win[1])

q90 fixed window: 2026-04-02 -> 2026-06-30
decision date t : 2026-05-31
straddles t -> future-contaminated: True


## Self-check

- [x] Every section states its thinking in markdown **and** backs it with an executed query
- [x] The notebook runs top to bottom with no errors (Kernel → Restart & Run All)
- [x] Grain verified on source *and* iteration slice; dedup rule written down where the grain broke
- [x] Every excluded field carries a one-line why; IDs stay context-only
- [x] No tokens, client names, URLs, or raw identifiers in any output — aggregates only
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to `work/notebooks/` — done right after this run passes